In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS houseprices.gold;

In [0]:
import gc
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.sql.types import DoubleType, IntegerType, LongType, FloatType, StringType

# ==========================================
# 1. CARGA E SEPARAÇÃO DINÂMICA DE COLUNAS
# ==========================================
# Carrega os dados perfeitamente limpos da Silver
df_silver = spark.table("houseprices.silver.silver_train")

coluna_target = "SalePrice"
colunas_ignoradas = ["Id", coluna_target]

# Identifica dinamicamente as colunas numéricas (removendo Id e Target)
colunas_numericas = [
    f.name for f in df_silver.schema.fields 
    if isinstance(f.dataType, (DoubleType, IntegerType, LongType, FloatType)) 
    and f.name not in colunas_ignoradas
]

# Identifica dinamicamente as colunas categóricas (textos)
colunas_categoricas = [
    f.name for f in df_silver.schema.fields 
    if isinstance(f.dataType, StringType) 
    and f.name not in colunas_ignoradas
]

# Cria os nomes das colunas que serão geradas nas etapas intermediárias do Spark ML
colunas_indexed = [c + "_indexed" for c in colunas_categoricas]
colunas_encoded = [c + "_encoded" for c in colunas_categoricas]

# ==========================================
# 2. CONFIGURAÇÃO DOS ESTÁGIOS DO PIPELINE (Spark ML)
# ==========================================

# Estágio A: StringIndexer (Mapeia strings para índices numéricos de 0 a N)
string_indexer = StringIndexer(
    inputCols=colunas_categoricas,
    outputCols=colunas_indexed,
    handleInvalid="keep"  # Nossa rede de segurança ativa para dados novos em produção
)

# Estágio B: OneHotEncoder (Transforma os índices em vetores binários / variáveis dummy)
one_hot_encoder = OneHotEncoder(
    inputCols=colunas_indexed,
    outputCols=colunas_encoded
)

# Estágio C: VectorAssembler (Consolida TODAS as variáveis em uma única coluna vetorial)
colunas_finais_features = colunas_numericas + colunas_encoded
vector_assembler = VectorAssembler(
    inputCols=colunas_finais_features,
    outputCol="features"
)

# ==========================================
# 3. EXECUÇÃO DO PIPELINE E MODELAGEM GOLD
# ==========================================
# Acopla todos os estágios na ordem correta de execução
pipeline = Pipeline(stages=[string_indexer, one_hot_encoder, vector_assembler])

# O Spark avalia o dataset e cria o modelo de transformação estruturado
pipeline_model = pipeline.fit(df_silver)
df_gold = pipeline_model.transform(df_silver)

# ==========================================
# 4. FILTRAGEM E SALVAMENTO DA TABELA DE CONSUMO
# ==========================================
# Isolamos apenas o Id (rastreabilidade), as Features vetorizadas e o Target (rótulo)
df_gold_final = df_gold.select("Id", "features", coluna_target)

# Grava o resultado final na tabela Gold no formato Delta
df_gold_final.write.format("delta").mode("overwrite").saveAsTable("houseprices.gold.gold_train_ml")

print("Camada Gold concluída com sucesso!")
display(df_gold_final.head(5))

# ==========================================
# 5. FAXINA DE MEMÓRIA (Evita erro SQLSTATE: 54000)
# ==========================================
# Deleta a referência do modelo atual e força a limpeza do cache do Spark Connect
del pipeline_model  
gc.collect()        
print("Memória do Spark ML limpa com sucesso!")